# Pandas ile Zaman Serisi ve Metin İşlemleri

Veri setlerinde tarih ve metin sütunları çok sık bulunur. Bu notebook'ta tarih dönüşümü, `dt` erişicisi, zaman bazlı gruplama ve metin temizleme işlemlerini öğreneceğiz.


## Konu Dokümantasyonu

Tarih verisi doğru türde değilse zaman analizi yapılamaz. Pandas'ta tarih sütunları `datetime64` türüne dönüştürülmelidir.

Tarih işlemleri için `dt` erişicisi kullanılır:

* `dt.year`
* `dt.month`
* `dt.day_name()`
* `dt.to_period()`

Metin işlemleri için `str` erişicisi kullanılır. Boşluk temizleme, küçük harfe çevirme, içerik arama ve parçalama işlemleri bu şekilde yapılır.


## Kolay Seviye

Tarih metinlerini `pd.to_datetime()` ile datetime türüne çevirelim.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "tarih": ["2026-01-05", "2026-01-12", "2026-02-03"],
    "satis": [1200, 1800, 2100],
})

df["tarih"] = pd.to_datetime(df["tarih"])
df["ay"] = df["tarih"].dt.month
df["gun_adi"] = df["tarih"].dt.day_name()

print(df)


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "tarih": pd.to_datetime(["2026-01-05", "2026-01-12", "2026-02-03"]),
    "satis": [1200, 1800, 2100],
})

df["yil_ay"] = df["tarih"].dt.to_period("M")
print(df)


## Orta Seviye

Zaman bazlı özetleme için tarih sütunu indeks yapılabilir ve `resample()` kullanılabilir.

Günlük veriden aylık toplam üretmek sık kullanılan bir zaman serisi işlemidir.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "tarih": pd.to_datetime(["2026-01-05", "2026-01-12", "2026-02-03", "2026-02-18"]),
    "satis": [1200, 1800, 2100, 2400],
})

aylik = (
    df.set_index("tarih")
    .resample("ME")["satis"]
    .sum()
)

print(aylik)


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "urun": [" Laptop ", "TELEFON", "akıllı saat", "Kulaklık"],
    "satis": [30000, 20000, 5000, 2500],
})

df["urun_temiz"] = df["urun"].str.strip().str.lower()
df["akilli_mi"] = df["urun_temiz"].str.contains("akıllı")

print(df)


## İleri Seviye

Tarih ve metin işlemleri birlikte kullanılarak analiz için yeni özellikler üretilebilir.

Örneğin sipariş tarihinden hafta sonu bilgisi, ürün adından kategori ipucu veya kampanya metninden etiket çıkarılabilir.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "tarih": pd.to_datetime(["2026-01-03", "2026-01-05", "2026-01-10", "2026-01-12"]),
    "kampanya": ["Yeni Yıl İndirimi", "Standart", "Hafta Sonu Fırsatı", "Standart"],
    "satis": [2200, 1200, 2800, 1500],
})

df["hafta_sonu_mu"] = df["tarih"].dt.weekday >= 5
df["kampanya_var_mi"] = df["kampanya"].str.lower().str.contains("indirim|fırsat", regex=True)

print(df)


## Mini Veri Bilimi Uygulaması

Sipariş verisinden aylık satış, hafta sonu etkisi ve kampanya etkisi için temel özetler çıkaralım.


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "tarih": pd.to_datetime(["2026-01-03", "2026-01-05", "2026-01-10", "2026-02-12", "2026-02-20"]),
    "kampanya": ["Yeni Yıl İndirimi", "Standart", "Hafta Sonu Fırsatı", "Standart", "Şubat Fırsatı"],
    "satis": [2200, 1200, 2800, 1500, 3200],
})

df["yil_ay"] = df["tarih"].dt.to_period("M")
df["kampanya_var_mi"] = df["kampanya"].str.lower().str.contains("indirim|fırsat", regex=True)

aylik_satis = df.groupby("yil_ay")["satis"].sum()
kampanya_ozeti = df.groupby("kampanya_var_mi")["satis"].mean()

print("Aylık satış:")
print(aylik_satis)
print("Kampanya ortalama satış:")
print(kampanya_ozeti)
